# Final Regression Model — European Cinema Admissions PredictorThis notebook trains the final Ridge regression model that predicts film performance (log z-score of admissions) across 35 European markets, using content features, language, origin country, genres, keywords, and budget bracket. It then exports the trained model as JSON for the interactive perturbation web app.

In [ ]:
import jsonfrom pathlib import Pathfrom collections import Counterimport numpy as npimport pandas as pdfrom scipy.sparse import csr_matrixfrom sklearn.linear_model import Ridgefrom sklearn.model_selection import train_test_splitfrom sklearn.metrics import mean_squared_error, r2_score

## 1. Load and merge data sourcesThree inputs: admissions data, content-only clusters, and IMDB budget data.

In [ ]:
PROJECT_ROOT = Path.cwd().parents[0]# Admissions data.df_admissions = pd.read_csv(PROJECT_ROOT / "Data" / "Lumiere_Country_Full_Backup.csv")# Content clusters (country/language excluded from clustering).df_clusters = pd.read_csv(    PROJECT_ROOT / "outputs" / "text_clustering"    / "results_content_only" / "clustered_films_k6.csv")# Budget data: combine films with budget info from both sheets.budget_both = pd.read_excel(PROJECT_ROOT / "Data" / "IMDB_financials.xlsx", sheet_name="Both")budget_only = pd.read_excel(PROJECT_ROOT / "Data" / "IMDB_financials.xlsx", sheet_name="Budget only")df_budget = pd.concat([budget_both, budget_only], ignore_index=True)df_budget = df_budget[["titleId", "budget_eur"]].dropna(subset=["budget_eur"])# Merge: admissions + clusters by row index, then budget by titleId.assert len(df_admissions) == len(df_clusters), "Row count mismatch!"df = df_admissions.copy()df["cluster"] = df_clusters["cluster"].valuesdf = df.merge(df_budget, on="titleId", how="inner")print(f"Admissions films: {len(df_admissions)}")print(f"Films with budget: {len(df_budget)}")print(f"Merged: {len(df)} films with admissions + clusters + budget")

## 2. Melt to long format and cleanEach row becomes a film × market country pair. We remove small countries, handle GB/IE overlap, compute the domestic flag, and create log z-scores.

In [ ]:
def parse_country_admissions(value):    if pd.isna(value) or value == "Not Found":        return {}    try:        return json.loads(value.replace('""', '"'))    except (json.JSONDecodeError, TypeError):        return {}records = []for idx, row in df.iterrows():    admissions = parse_country_admissions(row["country_admissions"])    for country, count in admissions.items():        if country.startswith("Total"):            continue        records.append({            "titleId": row["titleId"],            "originalTitle": row["originalTitle"],            "releaseYear": row["releaseYear"],            "mainCountry": row["mainCountry"],            "firstLanguage": row["firstLanguage"],            "genres": row["genres"],            "keywords": row["keywords"],            "cluster": row["cluster"],            "budget_eur": row["budget_eur"],            "country": country,            "admissions": count        })df_long = pd.DataFrame(records)# Remove countries with fewer than 50 films.MIN_FILMS = 50country_film_counts = df_long.groupby("country")["titleId"].nunique()small_countries = country_film_counts[country_film_counts < MIN_FILMS].index.tolist()df_long = df_long[~df_long["country"].isin(small_countries)].copy()# Remove redundant GB/IE rows when GB_IE exists for that film.films_with_gbie = set(df_long[df_long["country"] == "GB_IE"]["titleId"])mask_redundant = (    df_long["country"].isin(["GB", "IE"]) &    df_long["titleId"].isin(films_with_gbie))df_long = df_long[~mask_redundant].copy()# Domestic flag.def is_domestic(row):    origin = str(row["mainCountry"]).strip().upper()    market = str(row["country"]).strip().upper()    if market == origin:        return True    if market == "GB_IE" and origin in ("GB", "IE"):        return True    return Falsedf_long["is_domestic"] = df_long.apply(is_domestic, axis=1)# Target: log-transformed z-scores within each country.df_long["log_admissions"] = np.log1p(df_long["admissions"])df_long["admissions_zscore"] = (    df_long.groupby("country")["log_admissions"]    .transform(lambda x: (x - x.mean()) / x.std()))# Budget bracket.df_long["budget_bracket"] = pd.cut(    df_long["budget_eur"],    bins=[0, 3_000_000, 8_000_000, 25_000_000, float("inf")],    labels=["micro (<3M)", "european (3-8M)", "mid (8-25M)", "big (>25M)"])print(f"Long format: {len(df_long)} rows, {df_long['titleId'].nunique()} films, {df_long['country'].nunique()} countries")print(f"\nBudget bracket distribution (unique films):")print(df_long.groupby("budget_bracket")["titleId"].nunique())

## 3. Parse keywords and compute cross-country divisivenessKeywords appearing in 50+ films become base features. The top 15 most divisive keywords (highest cross-country variance in mean z-score) also get keyword × market interaction terms.

In [ ]:
def parse_keywords(value):    if pd.isna(value) or value == "":        return []    return [k.strip().lower() for k in str(value).split(",") if k.strip()]def parse_genres(value):    if pd.isna(value) or value == "":        return []    return [g.strip().lower() for g in str(value).split(",") if g.strip()]df_long["keywords_list"] = df_long["keywords"].apply(parse_keywords)df_long["genres_list"] = df_long["genres"].apply(parse_genres)# Keyword frequency across films.keyword_film_counts = (    df_long.drop_duplicates("titleId")    .explode("keywords_list")    ["keywords_list"].value_counts())# Compute keyword × country mean z-scores for divisiveness ranking.valid_keywords_50 = set(keyword_film_counts[keyword_film_counts >= 50].index)df_kw = df_long.explode("keywords_list")df_kw = df_kw[df_kw["keywords_list"].isin(valid_keywords_50)].copy()kw_country = (    df_kw.groupby(["keywords_list", "country"])["admissions_zscore"]    .agg(["mean", "count"])    .reset_index())kw_country = kw_country[kw_country["count"] >= 10].copy()kw_variance = (    kw_country.groupby("keywords_list")["mean"]    .agg(["std", "count"])    .rename(columns={"count": "n_countries"}))kw_variance = kw_variance[kw_variance["n_countries"] >= 15].sort_values("std", ascending=False)# Tier 1: keywords with 50+ films → base features.valid_keywords_base = valid_keywords_50# Tier 2: top 15 most divisive → keyword × market interactions.TOP_KW_INTERACTIONS = 15top_divisive_kw = kw_variance.head(TOP_KW_INTERACTIONS).index.tolist()print(f"Tier 1 base keywords (50+ films): {len(valid_keywords_base)}")print(f"Tier 2 interaction keywords (top {TOP_KW_INTERACTIONS} most divisive): {top_divisive_kw}")

## 4. Build feature matrixFeatures include: market country, origin country, language, content cluster, domestic flag, genres, keywords, budget bracket, and interaction terms (language × market, origin × market, cluster × market, cluster × domestic, keyword × market).

In [ ]:
y = df_long["admissions_zscore"].values# One-hot blocks.market_dummies = pd.get_dummies(df_long["country"], prefix="market")origin_dummies = pd.get_dummies(df_long["mainCountry"], prefix="origin")lang_dummies = pd.get_dummies(df_long["firstLanguage"], prefix="lang")cluster_dummies = pd.get_dummies(df_long["cluster"], prefix="cluster")domestic_flag = df_long["is_domestic"].astype(int).to_frame("is_domestic")bracket_dummies = pd.get_dummies(df_long["budget_bracket"], prefix="budget")# Genre dummies.all_genres = set()for g_list in df_long["genres_list"]:    all_genres.update(g_list)genre_dummies = pd.DataFrame(0, index=df_long.index, columns=[f"genre_{g}" for g in sorted(all_genres)])for i, g_list in enumerate(df_long["genres_list"]):    for g in g_list:        col = f"genre_{g}"        if col in genre_dummies.columns:            genre_dummies.iloc[i, genre_dummies.columns.get_loc(col)] = 1# Keyword dummies (Tier 1).kw_cols = sorted([f"kw_{k}" for k in valid_keywords_base])keyword_features = pd.DataFrame(0, index=df_long.index, columns=kw_cols, dtype=np.int8)for i, kw_list in enumerate(df_long["keywords_list"]):    for kw in kw_list:        col = f"kw_{kw}"        if col in keyword_features.columns:            keyword_features.iloc[i, keyword_features.columns.get_loc(col)] = 1print(f"Market:    {market_dummies.shape[1]}")print(f"Origin:    {origin_dummies.shape[1]}")print(f"Language:  {lang_dummies.shape[1]}")print(f"Cluster:   {cluster_dummies.shape[1]}")print(f"Genres:    {genre_dummies.shape[1]}")print(f"Keywords:  {keyword_features.shape[1]}")print(f"Brackets:  {bracket_dummies.shape[1]}")

In [ ]:
MIN_NONZERO = 20def drop_sparse_cols(df, min_nonzero=MIN_NONZERO):    keep = df.columns[(df != 0).sum(axis=0) >= min_nonzero]    return df[keep]market_vals = market_dummies.values# Cluster × domestic.cluster_domestic = pd.DataFrame(    cluster_dummies.values * domestic_flag.values,    columns=[f"{c}_x_domestic" for c in cluster_dummies.columns],    index=df_long.index)# Language × market.lang_market = pd.DataFrame(    (lang_dummies.values[:, :, None] * market_vals[:, None, :]).reshape(len(df_long), -1),    columns=[f"{l}_x_{m}" for l in lang_dummies.columns for m in market_dummies.columns],    index=df_long.index)lang_market = drop_sparse_cols(lang_market)# Origin × market.origin_market = pd.DataFrame(    (origin_dummies.values[:, :, None] * market_vals[:, None, :]).reshape(len(df_long), -1),    columns=[f"{o}_x_{m}" for o in origin_dummies.columns for m in market_dummies.columns],    index=df_long.index)origin_market = drop_sparse_cols(origin_market)# Cluster × market.cluster_market = pd.DataFrame(    (cluster_dummies.values[:, :, None] * market_vals[:, None, :]).reshape(len(df_long), -1),    columns=[f"{c}_x_{m}" for c in cluster_dummies.columns for m in market_dummies.columns],    index=df_long.index)cluster_market = drop_sparse_cols(cluster_market)# Keyword × market (Tier 2 only).kw_tier2_cols = [f"kw_{k}" for k in top_divisive_kw if f"kw_{k}" in keyword_features.columns]kw_tier2_matrix = keyword_features[kw_tier2_cols]kw_market = pd.DataFrame(    (kw_tier2_matrix.values[:, :, None] * market_vals[:, None, :]).reshape(len(df_long), -1),    columns=[f"{k}_x_{m}" for k in kw_tier2_cols for m in market_dummies.columns],    index=df_long.index)kw_market = drop_sparse_cols(kw_market)print(f"Language × market:  {lang_market.shape[1]}")print(f"Origin × market:    {origin_market.shape[1]}")print(f"Cluster × market:   {cluster_market.shape[1]}")print(f"Cluster × domestic: {cluster_domestic.shape[1]}")print(f"Keyword × market:   {kw_market.shape[1]}")

## 5. Train final modelWe compare two versions: base features + budget brackets (no interactions), and the full set with all interactions. We keep whichever generalizes better.

In [ ]:
# Split by film to prevent leakage.film_ids = df_long["titleId"].unique()train_films, test_films = train_test_split(film_ids, test_size=0.2, random_state=42)train_mask = df_long["titleId"].isin(train_films)test_mask = df_long["titleId"].isin(test_films)print(f"Train: {train_mask.sum()} rows ({len(train_films)} films)")print(f"Test:  {test_mask.sum()} rows ({len(test_films)} films)")# Version A: base features + brackets (no interactions).base_with_brackets = pd.concat([    market_dummies, origin_dummies, lang_dummies, cluster_dummies,    domestic_flag, genre_dummies, keyword_features, bracket_dummies], axis=1)# Version B: full features with interactions + brackets.full_with_interactions = pd.concat([    market_dummies, origin_dummies, lang_dummies, cluster_dummies,    domestic_flag, genre_dummies, keyword_features, bracket_dummies,    cluster_domestic, lang_market, origin_market, cluster_market, kw_market], axis=1)print(f"\nVersion A (base + brackets): {base_with_brackets.shape[1]} features")print(f"Version B (full + interactions): {full_with_interactions.shape[1]} features")# Test both across alpha values.for name, features_df in [("A: base+brackets", base_with_brackets), ("B: full+interactions", full_with_interactions)]:    Xtr = csr_matrix(features_df[train_mask].values.astype(np.float64))    Xte = csr_matrix(features_df[test_mask].values.astype(np.float64))        print(f"\n{name}:")    best_r2, best_alpha = -999, None    for alpha in [500, 1000, 2000, 3000, 5000]:        m = Ridge(alpha=alpha, solver="sparse_cg")        m.fit(Xtr, y[train_mask])        tr = m.score(Xtr, y[train_mask])        te = m.score(Xte, y[test_mask])        marker = ""        if te > best_r2:            best_r2 = te            best_alpha = alpha            marker = " <-- best"        print(f"  alpha={alpha:<6} Train R²={tr:.4f}  Test R²={te:.4f}{marker}")    print(f"  Best: alpha={best_alpha}, Test R²={best_r2:.4f}")

## 6. Refit the best modelSelect the best version and alpha from above, then refit on training data.

In [ ]:
# UPDATE THESE based on the results above.BEST_ALPHA = 2000.0USE_INTERACTIONS = True  # Set to False if version A was better.if USE_INTERACTIONS:    final_features = full_with_interactions    print("Using: full features with interactions")else:    final_features = base_with_brackets    print("Using: base features + brackets only")X_train_final = csr_matrix(final_features[train_mask].values.astype(np.float64))X_test_final = csr_matrix(final_features[test_mask].values.astype(np.float64))ridge_final = Ridge(alpha=BEST_ALPHA, solver="sparse_cg")ridge_final.fit(X_train_final, y[train_mask])train_r2 = ridge_final.score(X_train_final, y[train_mask])test_r2 = ridge_final.score(X_test_final, y[test_mask])test_rmse = np.sqrt(mean_squared_error(y[test_mask], ridge_final.predict(X_test_final)))print(f"\nFinal model: {final_features.shape[1]} features, alpha={BEST_ALPHA}")print(f"Train R²: {train_r2:.4f}")print(f"Test R²:  {test_r2:.4f}")print(f"Test RMSE: {test_rmse:.4f}")

## 7. Compute country stats for z-score → ticket conversionFor each country, store the mean and std of log-admissions so predictions can be converted from z-scores back to estimated ticket counts.

In [ ]:
country_stats = (    df_long.groupby("country")["log_admissions"]    .agg(["mean", "std", "count"])    .round(6))country_stats["mean_raw"] = df_long.groupby("country")["admissions"].mean().round(0)country_stats["median_raw"] = df_long.groupby("country")["admissions"].median().round(0)print("Country stats for z-score to ticket conversion:")print(country_stats)# Example conversion.print(f"\nExample: z-score of +0.5 in DE")de_mean = country_stats.loc["DE", "mean"]de_std = country_stats.loc["DE", "std"]log_est = 0.5 * de_std + de_meanticket_est = np.expm1(log_est)print(f"  log_admissions = 0.5 * {de_std:.3f} + {de_mean:.3f} = {log_est:.3f}")print(f"  estimated tickets = {ticket_est:,.0f}")

## 8. Export model for the web appSave everything the interactive perturbation tool needs: coefficients, feature names, country stats, and available options for each input field.

In [ ]:
def categorize_feature(name):    if "_x_market_" in name:        if name.startswith("lang_"):            return "language x market"        elif name.startswith("origin_"):            return "origin x market"        elif name.startswith("kw_"):            return "keyword x market"    if "_x_market_" in name and name.startswith("cluster_"):        return "cluster x market"    if "_x_domestic" in name:        return "cluster x domestic"    if name.startswith("market_"):        return "market baseline"    if name.startswith("origin_"):        return "origin"    if name.startswith("lang_"):        return "language"    if name.startswith("cluster_"):        return "cluster"    if name.startswith("genre_"):        return "genre"    if name.startswith("kw_"):        return "keyword"    if name.startswith("budget_"):        return "budget bracket"    if name == "is_domestic":        return "domestic flag"    return "other"# Build coefficient dict (skip near-zero values to keep file small).coef_dict = {}for feat, coef in zip(final_features.columns, ridge_final.coef_):    if abs(coef) > 0.0001:        coef_dict[feat] = round(float(coef), 6)export_data = {    "model_info": {        "alpha": BEST_ALPHA,        "n_features": int(final_features.shape[1]),        "n_nonzero_coefs": len(coef_dict),        "intercept": round(float(ridge_final.intercept_), 6),        "test_r2": round(test_r2, 4),        "n_train_films": len(train_films),        "n_test_films": len(test_films)    },    "coef_dict": coef_dict,    "countries": sorted(df_long["country"].unique().tolist()),    "country_stats": {        country: {            "mean_log": round(float(row["mean"]), 6),            "std_log": round(float(row["std"]), 6),            "mean_raw": float(row["mean_raw"]),            "median_raw": float(row["median_raw"]),            "n_films": int(row["count"])        }        for country, row in country_stats.iterrows()    },    "available_keywords": sorted([        col.replace("kw_", "") for col in final_features.columns        if col.startswith("kw_") and "_x_" not in col    ]),    "available_genres": sorted([        col.replace("genre_", "") for col in final_features.columns        if col.startswith("genre_")    ]),    "available_languages": sorted([        col.replace("lang_", "") for col in final_features.columns        if col.startswith("lang_")    ]),    "available_origins": sorted([        col.replace("origin_", "") for col in final_features.columns        if col.startswith("origin_")    ]),    "available_clusters": sorted([        int(col.replace("cluster_", ""))        for col in final_features.columns        if col.startswith("cluster_") and "_x_" not in col    ]),    "budget_brackets": sorted([        col.replace("budget_", "")        for col in final_features.columns        if col.startswith("budget_")    ]),    "cluster_descriptions": {        "0": "Horror / Thriller / Supernatural",        "1": "Serious Drama / War / Historical",        "2": "Buddy Comedy / Coming-of-age",        "3": "Crime Thriller / Detective",        "4": "Romance / Relationship Drama",        "5": "Adventure / Fantasy / Family"    },    "interaction_keywords": top_divisive_kw}# Save.EXPORT_DIR = PROJECT_ROOT / "outputs"EXPORT_DIR.mkdir(parents=True, exist_ok=True)export_path = EXPORT_DIR / "model_export.json"with open(export_path, "w") as f:    json.dump(export_data, f, indent=2)print(f"Exported to: {export_path}")print(f"File size: {export_path.stat().st_size / 1024:.0f} KB")print(f"\nContents:")print(f"  Non-zero coefficients: {len(coef_dict)}")print(f"  Countries: {len(export_data['countries'])}")print(f"  Keywords: {len(export_data['available_keywords'])}")print(f"  Genres: {len(export_data['available_genres'])}")print(f"  Languages: {len(export_data['available_languages'])}")print(f"  Origins: {len(export_data['available_origins'])}")print(f"  Interaction keywords: {len(export_data['interaction_keywords'])}")print(f"\n--- Upload model_export.json to the Claude chat to build the web app ---")